# The kernel playground — feel the three layers

This notebook is **running inside a kernel**: a separate Python process (an
IPython kernel, `ipykernel`) that this UI talks to over ZeroMQ. Nothing in
here is special notebook magic baked into the file — every output below was
produced by that live process. Let's prove it and poke at it.

> **Layered model:** `Jupyter (this UI)` ⇄ `ZeroMQ (signed JSON messages)` ⇄ `Kernel (IPython, holds your state)`

## 1. Which Python is my kernel? (the venv/`source` confusion)
The kernel is a *process*. The first thing worth knowing is which interpreter it is.

In [1]:
import sys, os
print("kernel interpreter :", sys.executable)
print("python version     :", sys.version.split()[0])
print("kernel process PID  :", os.getpid())

kernel interpreter : /Users/naledi/Projects/kernels-ipython-jupyter/.venv/bin/python
python version     : 3.14.4
kernel process PID  : 7258


That `sys.executable` is the source of endless confusion: the kernel points at
**one specific Python** (here, our `.venv`). Activating a different venv in your
terminal does **not** move a *running* kernel — the kernel keeps the interpreter
it was launched with. `!which python3` (the shell's idea) and `sys.executable`
(the kernel's reality) can disagree:

In [2]:
print("shell sees :", end=" "); 
!which python3

shell sees : /opt/homebrew/bin/python3


In [3]:
print("kernel is  :", sys.executable)

kernel is  : /Users/naledi/Projects/kernels-ipython-jupyter/.venv/bin/python


## 2. State lives in the kernel, not the cells
Cells aren't isolated scripts. They all run in the **same process**, so a name
defined in one cell is alive in the next. That persistence *is* the kernel.

In [4]:
secret = 6 * 7        # define it here, produce no output

In [5]:
secret * 2            # ...and it's still alive one cell later

84

## 3. `_`, `In`, `Out` — IPython's history (bare `python3` has none of this)
IPython caches every result. `_` is the last value; `Out[n]` is the value of cell *n*.

In [6]:
print("last result (_):", _)
print("Out cache keys     :", list(Out.keys()))

last result (_): 84
Out cache keys     : [5]


## 4. Magics — the `%`/`%%` commands that aren't Python
A *magic* is a command IPython intercepts before the line reaches Python.
`%timeit` measures, `%who` lists your variables, `%%writefile` saves a cell.

In [7]:
%timeit sum(range(10_000))

89.4 μs ± 29.2 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [8]:
name = 'naledi'
pi = 3.14159
%who          # every name currently alive in the kernel

No variables match your requested type.


### `%run` — execute a whole script *inside this kernel's namespace*

In [9]:
%%writefile hello_run.py
# This file is written by the cell above, then run by the cell below.
greeting = "hello from %run — I ran inside the kernel"
print(greeting)

Writing hello_run.py


In [10]:
%run hello_run.py
print("and `greeting` now lives in the notebook:", greeting)

hello from %run — I ran inside the kernel
and `greeting` now lives in the notebook: hello from %run — I ran inside the kernel


## 5. `!` — shell access straight from a cell
A leading `!` ships the line to the OS shell and pipes the output back. You can
even capture it into a Python variable.

In [11]:
!echo "shell says: I am $(uname -s) and the time is $(date +%H:%M:%S)"

shell says: I am Darwin and the time is 21:25:57


In [12]:
files = !ls -1
print("this cell captured a shell listing into Python:", files)

this cell captured a shell listing into Python: ['build_notebook.py', 'hello_run.py', 'playground.ipynb']


## 6. A custom magic — magics are just registered functions
`demos/custom_magic.py` defines `%clap` and `%%shout`. We load it and use it,
proving magics are extensible, not hard-coded.

In [13]:
import sys
sys.path.insert(0, "../demos")   # so the kernel can import custom_magic
%load_ext custom_magic
%clap kernels are just processes that hold state

'kernels 👏 are 👏 just 👏 processes 👏 that 👏 hold 👏 state'

In [14]:
%%shout
state persists across cells, and magics are yours to write

STATE PERSISTS ACROSS CELLS, AND MAGICS ARE YOURS TO WRITE


## 7. So what *is* each layer?

| Layer | What it is | In this notebook |
|---|---|---|
| **Kernel** | the process that runs code and holds state | the `ipykernel` PID printed in §1 |
| **IPython** | the enhanced REPL that the default kernel wraps | `_`, `Out[]`, magics, `!shell` above |
| **Jupyter** | the UI/front-end that talks to a kernel over ZeroMQ | the thing rendering this page |

Want to *see* the ZeroMQ messages this UI exchanges with the kernel?
Run `python demos/zmq_sniff.py` at the repo root — it prints the raw frames.